In [ ]:
from cosmograph import cosmo
from pathlib import Path
import pandas as pd
from backend.infrastructure.catalog.graph_snapshot import load_base_graph
def find_snapshot() -> Path:
    candidates = [
        root / "src/backend/infrastructure/fixtures/data/base_graph.pickle"
        for parent in (Path.cwd(), *Path.cwd().parents)
        for root in (parent, parent / "backend")
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find base_graph.pickle. Run this notebook from the repository "
        "root or the backend directory after generating the snapshot."
    )
snapshot_path = find_snapshot()
graph = load_base_graph(snapshot_path)
points = pd.DataFrame.from_records(
    [
        {
            "id": city_id,
            "label": f"{attributes['name']} ({attributes['country']})",
            "country": attributes["country"],
            "latitude": attributes["latitude"],
            "longitude": attributes["longitude"],
            "population": attributes["population"] or 0,
            "size": max((attributes["population"] or 0) ** 0.35, 1),
        }
        for city_id, attributes in graph.nodes(data=True)
    ]
)
route_rows = {}
for origin, destination, attributes in graph.edges(data=True):
    key = (origin, destination)
    route = route_rows.setdefault(
        key,
        {
            "source": origin,
            "target": destination,
            "connections": 0,
            "modes": set(),
            "distance_km": [],
            "travel_time_minutes": [],
        },
    )
    route["connections"] += 1
    route["modes"].add(attributes.get("mode", "UNKNOWN"))
    if attributes.get("distance_km") is not None:
        route["distance_km"].append(attributes["distance_km"])
    if attributes.get("travel_time_minutes") is not None:
        route["travel_time_minutes"].append(attributes["travel_time_minutes"])
links = pd.DataFrame.from_records(
    {
        "source": route["source"],
        "target": route["target"],
        "connections": route["connections"],
        "modes": ", ".join(sorted(route["modes"])),
        "distance_km": sum(route["distance_km"]) / len(route["distance_km"])
        if route["distance_km"]
        else 0,
        "travel_time_minutes": sum(route["travel_time_minutes"])
        / len(route["travel_time_minutes"])
        if route["travel_time_minutes"]
        else 0,
    }
    for route in route_rows.values()
)
cosmograph = cosmo(
    points=points,
    links=links,
    point_id_by="id",
    point_x_by="longitude",
    point_y_by="latitude",
    point_size_by="size",
    point_size_scale=0.01,
    point_size_range=[2, 24],
    point_color_by="country",
    point_label_by="label",
    link_source_by="source",
    link_target_by="target",
    link_width_by="connections",
    link_width_scale=0.4,
    link_width=0.5,
    link_arrows=True,
    simulation_gravity=0.1,
    simulation_center=1,
    space_size=1,
)

# Anchor only this widget to the viewport; leave notebook errors readable.
cosmograph.add_class("eu-graph-app")
cosmograph.layout.width = "100%"
cosmograph.layout.height = "100%"
cosmograph.layout.margin = "0"
cosmograph.layout.padding = "0"

from IPython.display import HTML, display

display(HTML("""
<style>
.eu-graph-app {
  position: fixed !important;
  inset: 0 !important;
  width: 100% !important;
  height: 100vh !important;
  height: 100dvh !important;
  max-width: none !important;
  margin: 0 !important;
  padding: 0 !important;
  overflow: hidden;
  z-index: 10;
}
/* Cosmograph ships a fixed 450px height on its internal wrapper. */
.eu-graph-app .wrapper {
  position: relative;
  width: 100% !important;
  height: 100% !important;
  overflow: hidden;
}
</style>
"""))
display(cosmograph)
